# Laboratory #4: Adversarial Learning and OOD Detection

In this laboratory session we will develop a methodology for detecting OOD samples and measuring the quality of OOD detection. We will also experiment with incorporating adversarial examples during training to render models more robust to adversarial attacks.

---
## Exercise 1: OOD Detection and Performance Evaluation
In this first exercise you will build a simple OOD detection pipeline and implement some performance metrics to evaluate its performance.

### Exercise 1.1: Build a simple OOD detection pipeline

Implement an OOD detection pipeline (like in the Flipped Activity notebook) using an ID and an OOD dataset of your choice. Some options:

+ CIFAR-10 (ID), Subset of CIFAR-100 (OOD). You will need to wrap CIFAR-100 in some way to select a subset of classes that are *not* in CIFAR-10 (see `torch.utils.data.Subset`).
+ Labeled Faces in the Wild (ID), CIFAR-10 or FakeData (OOD). The LfW dataset is available in Scikit-learn (see `sklearn.datasets.fetch_lfw_people`).
+ Something else, but if using images keep the images reasonably small!

In this exercise your *OOD Detector* should produce a score representing how "out of distribution" a test sample is. We will implement some metrics in the next exercise, but for now use the techniques from the flipped activity notebook to judge how well OOD scoring is working (i.e. histograms).

**Note**: Make sure you make a validation split of your ID dataset for testing.

#### Imports generali

In [1]:
import random
import torch
import torchinfo
from PIL import features
from torchvision.datasets import CIFAR10, CIFAR100
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.models import list_models, get_model
from torch import nn
from torch import optim
import numpy as np
from sklearn import metrics
import matplotlib.pyplot as plt
import pandas as pd
import torchvision
from transformers.quantizers.quantizer_hqq import weight

seed = 123

C:\Users\Utente\Desktop\DLA_HW\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'weight' from 'transformers.quantizers.quantizer_hqq' (C:\Users\Utente\Desktop\DLA_HW\.venv\Lib\site-packages\transformers\quantizers\quantizer_hqq.py)

#### Funzioni Utils

Funzione per configurare il file yaml e restituisce i parametri per la configurazione

In [ ]:
from types import SimpleNamespace
import yaml
import argparse

# funzione che quando viene richiamata restituisce i parametri di configurazione
def configurer():
    parser = argparse.ArgumentParser()
    parser.add_argument("config", help='YAML Configuration file')
    opts = yaml.load(open(parser.parse_args().config), Loader=yaml.Loader)
    opts = SimpleNamespace(**opts)
    opts.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(opts.device) # per sicurezza per vedere se non succede qualcosa alla gpu
    return opts

Funzione per il logging per eventuale training

In [ ]:
from rich.logging import RichHandler
import logging



def get_logger():
    FORMAT = "%(message)s"
    logging.basicConfig(
        level="NOTSET", format=FORMAT, datefmt="[%X]", handlers=[RichHandler()]
    )
    log = logging.getLogger("rich")
    return log


#LOG = get_logger()

Funzione per la visulazzazione struttura del modello e numero di parametri

In [ ]:
from torchinfo import summary
from rich.console import Console
console = Console()

def visualize(model, model_name, input_data):
    out = model(input_data)
    console.print(f'Computed output, shape = {out.shape=}')
    model_stats = summary(model,
                          input_data=input_data,
                          col_names=[
                              "input_size",
                              "output_size",
                              "num_params",
                          ],
                          row_settings=("var_names",),
                          col_width=18,
                          depth=8,
                          verbose=0,
                          )
    console.print(model_stats)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
device

#### Creiamo i dataset e i dataloader che ci interessano

vedere se riesco a risolvere questo problema... Ci mette 1h per scaricare 2 dataset altrimenti passerei a usare hugging face (_fixed problems with torchvision_)

In [ ]:
# from datasets import load_dataset
#
# cifar10_hf  = load_dataset("uoft-cs/cifar10")
# cifar100_hf = load_dataset("uoft-cs/cifar100")
#
# print(cifar10_hf)
# print(cifar100_hf)
#
# # .features["<col>"] è un oggetto ClassLabel, che espone .names (lista ordinata per indice)
# cifar10_classes  = cifar10_hf["train"].features["label"].names
# cifar100_classes = cifar100_hf["train"].features["fine_label"].names
#
# # Ricostruisco l'equivalente di class_to_idx
# cifar10_class_to_idx  = {name: idx for idx, name in enumerate(cifar10_classes)}
# cifar100_class_to_idx = {name: idx for idx, name in enumerate(cifar100_classes)}
#
# print("Cifar10: ", cifar10_class_to_idx )
# print("\n")
# print("Cifar100", cifar100_class_to_idx)

Presenta problemi scaricarli dai canali ufficiali rispetto a prenderli da hugging face tempi eccessivamente lunghi

In [ ]:
cifa10dt = CIFAR10("./data", train=True, download=True, transform=transforms.ToTensor())
cifa100dt = CIFAR100("./data", train=True, download=True, transform=transforms.ToTensor())
print(f"Cifar10", cifa10dt)
print(f"Cifar100", cifa100dt)

In [ ]:
print("Cifar10: ", cifa10dt.class_to_idx)
print("\n")
print("Cifar100", cifa100dt.class_to_idx)

Creiamo una classe dataset da wrapper per gestire i dati da i dataset di HF (_fixed problem with torchvision_)

In [ ]:
# from torch.utils.data import Dataset
# import torchvision.transforms.v2 as T
#
# class CifarDataset(Dataset):
#     def __init__(self, opts, data, img_col, label_col, transform=None):
#         self.opts = opts
#         self.data = data
#         self.img_col = img_col
#         self.label_col = label_col
#         if transform is None:
#             self.transform = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])
#
#     def __len__(self):
#         return len(self.data)
#
#     def __getitem__(self, idx):
#         features = self.data[idx]
#         labels = features[self.label_col]
#         images = features[self.img_col]
#         images = self.transform(images)
#
#         return {"images": images, "labels": labels}


Creiamo i dataloader (_fixed problem with torchvision_)

In [ ]:
# from torch.utils.data import DataLoader
#
# class MakeDataLoader:
#     def __init__(self, opts, HFdata, img_col, label_col, transform=None):
#         self.opts = opts
#         self.HFdata = HFdata
#
#         generator = torch.Generator().manual_seed(opts.seed)
#
#         train_dt = CifarDataset(opts, HFdata["train"], img_col, label_col, transform=transform)
#
#         train, val = torch.utils.data.random_split(train_dt, lengths=[1 - opts.val_size, opts.val_size],
#                                                     generator=generator)
#         self.train_dl = DataLoader(train, batch_size=opts.batch_size, shuffle=True, num_workers=opts.num_workers)
#         self.val_dl = DataLoader(val, batch_size=opts.batch_size, shuffle=False, num_workers=opts.num_workers)
#
#         test_dt = CifarDataset(opts, HFdata["test"], img_col, label_col)
#         self.test_dl = DataLoader(test_dt, batch_size=opts.batch_size, shuffle=False, num_workers=opts.num_workers)



Funzione per caricare un checkpoint

In [ ]:
def load_checkpoint(model: nn.Module, checkpoint_path: str, device="cpu") -> nn.Module:
    """
    Carica i pesi (state_dict) salvati all'interno di un modello PyTorch.

    Args:
        model: L'istanza dell'architettura del modello vuota (es. SimpleCNN()).
        checkpoint_path: Il percorso del file .pth (es. 'SimpleCNN_best.pth').
        device: 'cpu' o 'cuda', per mappare correttamente i pesi.

    Returns:
        Il modello con i pesi caricati, già impostato in modalità eval().
    """
    # 1. Controllo di sicurezza sull'esistenza del file
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"️ Impossibile trovare il checkpoint: {checkpoint_path}. Assicurati che il nome sia corretto.")

    # 2. Caricamento del dizionario dei pesi
    # map_location=device è cruciale: permette di caricare su CPU un modello allenato su GPU (e viceversa) senza crash.
    state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)

    # 3. Iniezione dei pesi nel modello
    model.load_state_dict(state_dict)

    # 4. Spostamento sul device corretto e setup per l'inferenza
    model = model.to(device)
    model.eval() # Fondamentale: spegne il Dropout e blocca le BatchNorm in modalità test

    print(f"Checkpoint '{checkpoint_path}' caricato con successo sul device: {device}")

    return model

Creiamo le funzioni transform (vorrei trattare Vit, CNN casuale, ResNet18)

In [ ]:
ViT_transform = transforms.Compose([
    transforms.Resize(224),  # Resize to match ViT input
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))  # CIFAR-10 stats
])

CNNtransform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

ResNet_transform =transforms.Compose( # pretrained resnet (torchvision)
    [transforms.Resize(224),
    transforms.ToTensor(),
     transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))])

Creiamo una Semplice CNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2, 2)

        self.gap = nn.AdaptiveAvgPool2d(1)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x): # [B, 3, 32, 32] [B, C, H, W]
        # Primo blocco convoluzionale
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x) # Riduce a [B, 64, 16, 16]

        # Secondo blocco convoluzionale
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.pool(x) # Riduce a [B, 256, 8, 8]

        # Classificatore
        x = self.gap(x) # [B, 256, 1, 1]
        x = self.flatten(x) # [B, 256]
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x



Creiamo la funzione di training/finetuning

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import torch.nn.functional as F
import torch
import numpy as np
from torch.utils.tensorboard import SummaryWriter
import os

def train_epoch(model: nn.Module, dl: torch.utils.data.DataLoader , optimizer: torch.optim.Optimizer, epoch ="Unknown", device ="cpu"):
    model.train()
    losses = []
    for (xs, ys) in tqdm(dl, desc=f"Epoch {epoch}", leave=True):
        xs = xs.to(device)
        ys = ys.to(device)
        optimizer.zero_grad()
        out = model(xs)
        loss = F.cross_entropy(out, ys)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return np.mean(losses)

@torch.no_grad()
def evaluate_model(model: nn.Module, dl: torch.utils.data.DataLoader , device="cpu"):
    model.eval()
    preds = []
    gts = []
    with torch.no_grad():
        for (xs, ys) in tqdm(dl, desc=f"Evaluating", leave=False):
            xs = xs.to(device)
            pred = torch.argmax(model(xs), dim=1)
            gts.append(ys)
            preds.append(pred.detach().cpu().numpy())

        #return accuracy score and classification report
        return (accuracy_score(np.hstack(gts), np.hstack(preds))), classification_report(np.hstack(gts), np.hstack(preds), zero_division=0, digits=1)


def main_train(model: nn.Module, dl_train: torch.utils.data.DataLoader, dl_val: torch.utils.data.DataLoader, epochs: int, model_name: str, lr: float = 1e-3, device="cpu"):
    # 1. Setup Logger e TensorBoard
    LOG = get_logger()
    writer = SummaryWriter(log_dir=os.path.join("runs", model_name))

    LOG.info(f"Inizializzazione training per il modello: {model_name}")

    # 2. Sposta il modello sul device corretto
    model = model.to(device)

    # 3. Visualizzazione Struttura Modello (CIFAR-10 usa immagini 3x32x32)
    LOG.info("Generazione summary del modello...")
    dummy_input = torch.randn(1, 3, 32, 32).to(device)
    visualize(model, model_name, dummy_input)

    # 4. Inizializzazione Ottimizzatore (puoi passarlo come parametro se preferisci)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_acc = 0.0

    # 5. Loop di Addestramento
    LOG.info(f"Inizio addestramento per {epochs} epoche sul device: {device}")
    for epoch in range(1, epochs + 1):

        # --- TRAINING ---
        train_loss = train_epoch(model, dl_train, optimizer, epoch, device)

        # --- VALIDATION ---
        val_acc, val_report = evaluate_model(model, dl_val, device)

        # --- LOGGING TERMINALE (Rich) ---
        LOG.info(f"Epoch {epoch}/{epochs} completata!")
        LOG.info(f"Train Loss: {train_loss:.4f} | Val Accuracy: {val_acc:.4f}")

        # --- LOGGING TENSORBOARD ---
        writer.add_scalar("Loss/Train", train_loss, epoch)
        writer.add_scalar("Accuracy/Validation", val_acc, epoch)

        # --- SALVATAGGIO BEST MODEL ---
        if val_acc > best_acc:
            best_acc = val_acc
            save_path = f"{model_name}_best.pth"
            torch.save(model.state_dict(), save_path)
            LOG.info(f"Nuovo best model salvato in {save_path} con accuracy: {best_acc:.4f}")

    # Log Finale del Classification Report per avere un'idea dettagliata delle performance
    LOG.info("\nClassification Report Finale (Best Epochs escluse):\n" + val_report)
    LOG.info("Training concluso! ")


    # Chiudi il writer di TensorBoard
    writer.close()

    # carichiamo il miglior modello
    model.load_state_dict(torch.load(save_path))
    return model



Funzione per ottenere le classi più lontani OOD di CIFAR100 rispetto a CIFAR10

In [ ]:
from torch.utils.data import Subset

def get_far_ood_subset(dataset: torchvision.datasets.CIFAR100, classes_to_keep: list) -> Subset:
    """
    Estrae un sottoinsieme (Subset) da un dataset, mantenendo SOLO le classi specificate.
    Far-OOD.

    Args:
        dataset: Il dataset originale (es. CIFAR-100 test set).
        classes_to_keep: Lista di stringhe con i nomi delle classi da conservare.

    Returns:
        Un oggetto torch.utils.data.Subset pronto per essere passato a un DataLoader.
    """
    # 1. Recuperiamo il dizionario interno del dataset che mappa nomi -> numeri
    class_dict = dataset.class_to_idx

    # 2. Convertiamo i nomi in ID numerici (ignorando eventuali nomi scritti male)
    valid_ids = [class_dict[name] for name in classes_to_keep if name in class_dict]

    if not valid_ids:
        raise ValueError("Nessuna delle classi specificate è presente nel dataset!")

    # 3. Estraiamo tutte le etichette originali
    targets = np.array(dataset.targets)

    # 4. Creiamo la maschera logica e troviamo le posizioni esatte da conservare
    mask = np.isin(targets, valid_ids)
    indices_to_keep = np.where(mask)[0].tolist()

    print(f"Filtro applicato: mantenute {len(valid_ids)} classi su 100.")
    print(f"Dimensione dataset originale: {len(targets)} -> Nuovo OOD Subset: {len(indices_to_keep)}")

    # 5. Restituiamo il Subset pacchettizzato
    return Subset(dataset, indices_to_keep)

In [ ]:
list_models()

Funzione di generazione dataloaders ID e OOD

In [ ]:
def generate_dataloaders(transform, batch_size= 256, split= 0.3):
    # Dataset training validation testing ID
    ID_train_dt = CIFAR10("./data", train=True, download=True, transform=transform)
    ID_test_dt = CIFAR10("./data", train=False, download=True, transform=transform)

    generator = torch.Generator().manual_seed(seed)
    train, val  = torch.utils.data.random_split(ID_train_dt, lengths=[1 - split, split],  generator=generator)

    ID_tr_dl = DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True)
    ID_val_dl = DataLoader(val, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True)
    ID_ts_dl = DataLoader(ID_test_dt, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True)
    # Dataset testing OOD
    OOD_dt = CIFAR100("./data", train=False, download=True, transform=ViT_transform)

    # scegliamo le classi più lontane da ID e creiamo un Subset

    clean_ood_classes = [
        'orchid', 'poppy', 'rose', 'sunflower', 'tulip',       # Fiori
        'bottle', 'bowl', 'can', 'cup', 'plate',               # Contenitori
        'maple_tree', 'oak_tree', 'palm_tree', 'pine_tree'     # Alberi
    ]

    OOD_dt = get_far_ood_subset(OOD_dt, clean_ood_classes)

    OOD_dl = DataLoader(OOD_dt, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True)
    return {"train": ID_tr_dl, "val": ID_val_dl, "test": ID_ts_dl, "OOD": OOD_dl}

Funzione per generare il modello da finetunare

In [ ]:
def get_model_for_cifar10(model_name, device="cpu"):
    model = get_model(model_name, weights="DEFAULT")  # o weights=None per training da zero
    model.to(device)

    # Recupera in_features in base al tipo di modello, e sostituisce l'head con Identity
    if model_name == "resnet18":
        in_features = model.fc.in_features
        model.fc = nn.Identity()

    elif model_name == "vit_b_16":
        in_features = model.heads.head.in_features
        model.heads = nn.Identity()

    # Crea la head MLP
    head = MLPHead(in_features=in_features, num_classes=10, hidden_dim=256, dropout=0.3)

    # Modello finale = backbone + head MLP
    full_model = nn.Sequential(model, head).to(device)
    return full_model

In [ ]:
import torch.nn as nn

class MLPHead(nn.Module):
    def __init__(self, in_features, num_classes, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.mlp(x)

---
### Fase di Training-Tuning

In [ ]:
# -----------------------------
# PARAMETRI
model_name= "vit_b_16"
batch_size= 256
epochs = 10
lr = 1e-3
#----------------------

data = generate_dataloaders(transform=ViT_transform, batch_size=batch_size, split=0.3)

train_dl = data["train"]
val_dl = data["val"]
test_dl = data["test"]
OOD_dl = data["OOD"]

model = get_model_for_cifar10(model_name, device)

model = main_train(model, train_dl, val_dl, epochs, model_name, lr, device)

test_results = evaluate_model(model, test_dl, device)

print(test_results)



In [ ]:
# -----------------------------
# PARAMETRI
model_name= "resnet18"
batch_size= 1024
epochs = 10
lr = 1e-3
#----------------------

data = generate_dataloaders(transform=ResNet_transform, batch_size=batch_size, split=0.3)

train_dl = data["train"]
val_dl = data["val"]
test_dl = data["test"]
OOD_dl = data["OOD"]

model = get_model_for_cifar10(model_name, device)

model = main_train(model, train_dl, val_dl, epochs, model_name, lr, device)

test_results = evaluate_model(model, test_dl, device)

print(test_results)

In [ ]:
# -----------------------------
# PARAMETRI
model_name= "simplecnn"
batch_size= 1024
epochs = 15
lr = 0.01
#----------------------

data = generate_dataloaders(transform=ResNet_transform, batch_size=batch_size, split=0.3)

train_dl = data["train"]
val_dl = data["val"]
test_dl = data["test"]
OOD_dl = data["OOD"]

#model = get_model_for_cifar10(model_name, device)

model = SimpleCNN()

model = main_train(model, train_dl, val_dl, epochs, model_name, lr, device)

test_results = evaluate_model(model, test_dl, device)

print(test_results)

### Exercise 1.2: Measure your OOD detection performance

There are several metrics used to evaluate OOD detection performance, we will concentrate on two threshold-free approaches: the area under the Receiver Operator Characteristic (ROC) curve for ID classification, and the area under the Precision-Recall curve for *both* ID and OOD scoring. See [the ODIN paper](https://arxiv.org/pdf/1706.02690.pdf) section 4.3 for a description of OOD metrics.

Use the functions in `sklearn.metrics` to produce ROC and PR curves for your OOD detector. Some useful functions:

+ [`sklearn.metric.RocCurveDisplay.from_predictions`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.RocCurveDisplay.html)
+ [`sklearn.metrics.PrecisionRecallDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.PrecisionRecallDisplay.html)


In [ ]:
# Your code here.

---
## Exercise 2: Enhancing Robustness to Adversarial Attack

In this second exercise we will experiment with enhancing our base model to be (more) robust to adversarial attacks. 

### Exercise 2.1: Implement FGSM and generate adversarial examples

Recall that the Fast Gradient Sign Method (FGSM) perturbs samples in the direction of the gradient with respect to the input $\mathbf{x}$:
$$ \boldsymbol{\eta}(\mathbf{x}) = \varepsilon \mathrm{sign}(\nabla_{\mathbf{x}} \mathcal{L}(\boldsymbol{\theta}, \mathbf{x}, y)) ) $$
Implement FGSM and generate some *adversarial examples* using your trained ID model. Evaluate these samples qualitatively and quantitatively. Evaluate how dependent on $\varepsilon$ the quality of these samples are. 

In [ ]:
# Your code here.

### Exercise 2.2: Augment training with adversarial examples

Use your implementation of FGSM to augment your training dataset with adversarial samples. Ideally, you should implement this data augmentation *on the fly* so that the adversarial samples are always generated using the current model. Evaluate whether the model is more (or less) robust to ID samples using your OOD detection pipeline and metrics you implemented in Exercise 1.

In [ ]:
# Your code here.

---
## Exercise 3: Wildcard

You know the drill. Pick *ONE* of the following exercises to complete.

### Exercise 3.1: Implement ODIN for OOD detection
ODIN is a very simple approach, and you can already start experimenting by implementing a temperature hyperparameter in your base model and doing a grid search on $T$ and $\varepsilon$.

### Exercise 3.2: Implement JARN
In exercise 2.2 you already implemented Jacobian-regularized learning to make your model more robust to adversarial samples. Add a *discriminator* to your model to encourage the adversarial samples used for regularization to be more *salient*.

See [the JARN paper](https://arxiv.org/abs/1912.10185) for more details.

### Exercise 3.3: Experiment with *targeted* adversarial attacks
Implement the targeted Fast Gradient Sign Method to generate adversarial samples that *imitate* samples from a specific class. Evaluate your adversarial samples qualitatively and quantitatively.
